# 07 — Output vs Latent Constraints

**Context Demystifies Forecasting**

This notebook tests the repo's first experimental question:

> Does constraining hidden state outperform correcting outputs afterward?

We compare three toy forecasting modes:

1. **Baseline rollout** — forecast from recent history with no correction.
2. **Output-constrained rollout** — forecast first, then correct the output.
3. **Latent-constrained rollout** — estimate physical state, evolve it with a constrained transition, then decode.

This is not a reproduction of Phys-JEPA. It is a minimal experiment that makes the paper's core engineering idea visible.


## Setup

We create a toy system with:

```text
observed = physical process + residual variation
```

The physical process has a periodic component plus a slow trend. The residual variation is small and irregular.

This lets us test whether forecasts are more stable when we constrain the internal state transition instead of only correcting outputs afterward.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")

In [ ]:
n = 900
t = np.linspace(0, 18 * np.pi, n)
dt = t[1] - t[0]

periodic = np.sin(t)
trend = 0.012 * t
physical = periodic + trend
residual = 0.08 * rng.normal(size=n)

observed = physical + residual

df = pd.DataFrame({
    "t": t,
    "periodic": periodic,
    "trend": trend,
    "physical": physical,
    "residual": residual,
    "observed": observed,
})

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["t"], df["observed"], label="observed")
ax.plot(df["t"], df["physical"], label="physical process")

ax.set_title("Toy system: observed values and physical process")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "07_toy_system.png", dpi=160)
plt.show()

## Forecast task

We use the first part of the sequence as context and forecast the remaining horizon.

All three methods start from the same observed history.

The important comparison is not short-term accuracy. It is how error accumulates as the forecast horizon grows.


In [ ]:
train_end = 620
forecast_horizon = n - train_end

history_t = t[:train_end]
future_t = t[train_end:]

history_y = observed[:train_end]
truth_y = observed[train_end:]
truth_physical = physical[train_end:]

print(f"history length: {len(history_y)}")
print(f"forecast horizon: {forecast_horizon}")

## Method 1 — Baseline rollout

The baseline estimates the most recent value and slope from observed data, then rolls forward.

This is intentionally simple. It shows how a forecast can drift when it has no internal model of the physical transition.


In [ ]:
def baseline_rollout(history, horizon, window=18):
    history = np.asarray(history)
    recent = history[-window:]
    x = np.arange(window)

    slope, intercept = np.polyfit(x, recent, deg=1)
    start = recent[-1]

    steps = np.arange(1, horizon + 1)
    forecast = start + slope * steps
    return forecast

baseline_forecast = baseline_rollout(history_y, forecast_horizon)

## Method 2 — Output-constrained rollout

The output-constrained method predicts first, then applies a correction afterward.

Here the correction is a simple range and smoothness constraint estimated from the training history.

This represents a common pattern:

```text
history → prediction → correction
```


In [ ]:
def output_constrained_rollout(history, horizon, window=18, smooth_weight=0.85):
    raw = baseline_rollout(history, horizon, window=window)

    # Range estimated from the observed history.
    lower = np.quantile(history, 0.02)
    upper = np.quantile(history, 0.98)

    corrected = np.clip(raw, lower, upper)

    # Smooth the corrected outputs after prediction.
    smoothed = corrected.copy()
    for i in range(1, len(smoothed)):
        smoothed[i] = smooth_weight * smoothed[i - 1] + (1 - smooth_weight) * smoothed[i]

    return smoothed

output_forecast = output_constrained_rollout(history_y, forecast_horizon)

## Method 3 — Latent-constrained rollout

The latent-constrained method estimates a structured physical state from the history and evolves that state forward.

For this toy process, the physical state is represented by:

```text
periodic component + trend component
```

This is the simplified analogue of applying constraints to latent state transitions before decoding.


In [ ]:
def estimate_period_and_trend(history_t, history_y):
    # Estimate trend by linear regression.
    slope, intercept = np.polyfit(history_t, history_y, deg=1)
    detrended = history_y - (slope * history_t + intercept)

    # Estimate phase using the known toy frequency basis.
    sin_basis = np.sin(history_t)
    cos_basis = np.cos(history_t)
    X = np.column_stack([sin_basis, cos_basis, np.ones_like(history_t)])
    coef, *_ = np.linalg.lstsq(X, detrended, rcond=None)

    a_sin, b_cos, offset = coef
    amplitude = np.sqrt(a_sin ** 2 + b_cos ** 2)
    phase = np.arctan2(b_cos, a_sin)

    return {
        "slope": slope,
        "intercept": intercept,
        "amplitude": amplitude,
        "phase": phase,
        "offset": offset,
    }

def latent_constrained_rollout(history_t, history_y, future_t):
    state = estimate_period_and_trend(history_t, history_y)

    periodic_forecast = state["amplitude"] * np.sin(future_t + state["phase"])
    trend_forecast = state["slope"] * future_t + state["intercept"]
    forecast = periodic_forecast + trend_forecast + state["offset"]

    return forecast, state

latent_forecast, latent_state = latent_constrained_rollout(history_t, history_y, future_t)

latent_state

## Compare forecasts

We now compare all three rollouts over the same horizon.


In [ ]:
forecast_df = pd.DataFrame({
    "t": future_t,
    "truth": truth_y,
    "baseline": baseline_forecast,
    "output_constrained": output_forecast,
    "latent_constrained": latent_forecast,
})

forecast_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(forecast_df["t"], forecast_df["truth"], label="truth")
ax.plot(forecast_df["t"], forecast_df["baseline"], label="baseline")
ax.plot(forecast_df["t"], forecast_df["output_constrained"], label="output-constrained")
ax.plot(forecast_df["t"], forecast_df["latent_constrained"], label="latent-constrained")

ax.set_title("Output constraints vs latent constraints")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "07_output_vs_latent_constraints.png", dpi=160)
plt.show()

## Metrics

We compare mean absolute error and root mean squared error over the full rollout.


In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

metrics = pd.DataFrame([
    {
        "method": "baseline",
        "MAE": mae(forecast_df["truth"], forecast_df["baseline"]),
        "RMSE": rmse(forecast_df["truth"], forecast_df["baseline"]),
    },
    {
        "method": "output_constrained",
        "MAE": mae(forecast_df["truth"], forecast_df["output_constrained"]),
        "RMSE": rmse(forecast_df["truth"], forecast_df["output_constrained"]),
    },
    {
        "method": "latent_constrained",
        "MAE": mae(forecast_df["truth"], forecast_df["latent_constrained"]),
        "RMSE": rmse(forecast_df["truth"], forecast_df["latent_constrained"]),
    },
])

metrics.to_csv(RESULTS / "07_metrics.csv", index=False)
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

x = np.arange(len(metrics))
ax.bar(x, metrics["RMSE"])
ax.set_xticks(x)
ax.set_xticklabels(metrics["method"], rotation=20, ha="right")

ax.set_title("Full-horizon RMSE")
ax.set_ylabel("RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "07_full_horizon_rmse.png", dpi=160)
plt.show()

## Long-horizon error accumulation

The central question is how quickly forecast error grows over time.

We compute cumulative RMSE as horizon increases.


In [ ]:
def cumulative_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    sq_error = (y_true - y_pred) ** 2
    return np.sqrt(np.cumsum(sq_error) / np.arange(1, len(sq_error) + 1))

forecast_df["baseline_cumulative_rmse"] = cumulative_rmse(forecast_df["truth"], forecast_df["baseline"])
forecast_df["output_cumulative_rmse"] = cumulative_rmse(forecast_df["truth"], forecast_df["output_constrained"])
forecast_df["latent_cumulative_rmse"] = cumulative_rmse(forecast_df["truth"], forecast_df["latent_constrained"])

forecast_df.to_csv(RESULTS / "07_forecasts.csv", index=False)

forecast_df[
    [
        "t",
        "baseline_cumulative_rmse",
        "output_cumulative_rmse",
        "latent_cumulative_rmse",
    ]
].head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

horizon = np.arange(1, len(forecast_df) + 1)

ax.plot(horizon, forecast_df["baseline_cumulative_rmse"], label="baseline")
ax.plot(horizon, forecast_df["output_cumulative_rmse"], label="output-constrained")
ax.plot(horizon, forecast_df["latent_cumulative_rmse"], label="latent-constrained")

ax.set_title("Error accumulation across forecast horizon")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("cumulative RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "07_horizon_error.png", dpi=160)
plt.show()

## Physical state and residual state

The latent-constrained model can expose an interpretable state estimate.

Here we compare the true toy physical process with the estimated latent-constrained physical forecast.


In [ ]:
estimated_physical = latent_forecast
estimated_residual = truth_y - estimated_physical

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(future_t, truth_physical, label="true physical process")
ax.plot(future_t, estimated_physical, label="estimated constrained physical state")
ax.plot(future_t, estimated_residual, label="residual after latent forecast")

ax.set_title("Latent-constrained decomposition")
ax.set_xlabel("time")
ax.set_ylabel("component value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "07_latent_constraint.png", dpi=160)
plt.show()

## Interpretation

In this toy setting:

```text
Output constraints repair forecasts.
Latent constraints shape forecasts.
```

The output-constrained method corrects the prediction after the model has already drifted.

The latent-constrained method imposes structure before decoding future values.

That is the engineering distinction this repo will keep testing:

> Put constraints where the model represents the system, not only where it emits predictions.


## Next

Notebook 13 will make the central decomposition explicit:

```text
latent_state = physical_state + residual_state
```

The goal will be to move from an observable toy decomposition toward a reusable latent-state decomposition workflow.
